# The 30-day window: time to second order in e-commerce

**Question:** what separates a low-value new customer from a high-value one in their first 6 months?

**Data:** [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail): all transactions of a UK-based online retailer, Dec 2010 to Dec 2011 (CC BY 4.0).

**Approach:**
1. Clean transactions (drop cancellations, returns, zero prices, rows without a customer ID).
2. Build a cohort of customers whose first order was before 1 June 2011, so every customer has at least 6 months of observation.
3. Segment customers by days between their 1st and 2nd order.
4. Compare each segment's share of customers, share of 6-month revenue, and median revenue per customer.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
COHORT_END = pd.Timestamp("2011-06-01")   # first orders before this date
WINDOW_DAYS = 180                          # 6-month observation window

## 1. Load and clean

In [2]:
def load_raw(data_dir: Path) -> pd.DataFrame:
    """Load the official UCI Excel file if present, otherwise a CSV export of it."""
    xlsx = data_dir / "Online Retail.xlsx"
    if xlsx.exists():
        return pd.read_excel(xlsx)
    csv = data_dir / "Online_Retail.csv"
    with open(csv, "rb") as fh:
        sample = fh.read(4096)
    lt = "\r" if b"\r" in sample and b"\n" not in sample else None
    df = pd.read_csv(csv, encoding="latin1", lineterminator=lt)
    df.columns = df.columns.str.strip()
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="mixed")
    return df

raw = load_raw(DATA_DIR)
print(f"Raw rows: {len(raw):,}")

df = raw.dropna(subset=["CustomerID"]).copy()
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]   # cancellations
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print(f"Clean rows: {len(df):,}")
print(f"Customers: {df['CustomerID'].nunique():,}")
print(f"Period: {df['InvoiceDate'].min():%Y-%m-%d} to {df['InvoiceDate'].max():%Y-%m-%d}")

Raw rows: 541,909
Clean rows: 397,884
Customers: 4,338
Period: 2010-12-01 to 2011-12-09


## 2. Order-level table and the cohort

In [3]:
orders = (df.groupby(["CustomerID", "InvoiceNo"])
            .agg(order_time=("InvoiceDate", "min"), revenue=("Revenue", "sum"))
            .reset_index()
            .sort_values("order_time"))
orders["order_number"] = orders.groupby("CustomerID").cumcount() + 1

first = orders[orders["order_number"] == 1].set_index("CustomerID")
second = orders[orders["order_number"] == 2].set_index("CustomerID")

customers = pd.DataFrame({
    "first_order_time": first["order_time"],
    "first_order_value": first["revenue"],
    "days_to_second": (second["order_time"] - first["order_time"]).dt.days,
})

cohort = customers[customers["first_order_time"] < COHORT_END].copy()

o = orders.merge(cohort[["first_order_time"]], left_on="CustomerID", right_index=True)
in_window = o["order_time"] <= o["first_order_time"] + pd.Timedelta(days=WINDOW_DAYS)
cohort["revenue_6m"] = o[in_window].groupby("CustomerID")["revenue"].sum()

print(f"Cohort size: {len(cohort):,} customers")

Cohort size: 2,718 customers


## 3. Segment by time to second order

In [4]:
def segment(days):
    if pd.isna(days) or days > WINDOW_DAYS:
        return "No 2nd order"
    if days <= 30:
        return "<=30 days"
    if days <= 90:
        return "31-90 days"
    return "91-180 days"

order = ["<=30 days", "31-90 days", "91-180 days", "No 2nd order"]
cohort["segment"] = pd.Categorical(cohort["days_to_second"].map(segment), categories=order, ordered=True)

summary = cohort.groupby("segment", observed=True).agg(
    customers=("revenue_6m", "size"),
    revenue_6m=("revenue_6m", "sum"),
    median_revenue_6m=("revenue_6m", "median"),
    median_first_order=("first_order_value", "median"),
)
summary["share_of_customers"] = summary["customers"] / summary["customers"].sum()
summary["share_of_revenue"] = summary["revenue_6m"] / summary["revenue_6m"].sum()
summary.style.format({
    "revenue_6m": "£{:,.0f}", "median_revenue_6m": "£{:,.0f}", "median_first_order": "£{:,.0f}",
    "share_of_customers": "{:.1%}", "share_of_revenue": "{:.1%}",
})

,customers,revenue_6m,median_revenue_6m,median_first_order,share_of_customers,share_of_revenue
segment,,,,,,
<=30 days,646,"£2,086,522","£1,515",£314,23.8%,52.5%
31-90 days,685,"£1,054,266","£1,063",£315,25.2%,26.5%
91-180 days,482,"£460,804",£694,£304,17.7%,11.6%
No 2nd order,905,"£370,081",£233,£233,33.3%,9.3%


## 4. Timing of the second order

In [5]:
repeat = cohort.loc[cohort["days_to_second"] <= WINDOW_DAYS, "days_to_second"]
print(f"Placed a 2nd order within {WINDOW_DAYS} days: {len(repeat)/len(cohort):.1%}")
for d in (30, 60, 90):
    print(f"  of those, within {d} days: {(repeat <= d).mean():.1%}")
print(f"Median days to 2nd order: {repeat.median():.0f}")

Placed a 2nd order within 180 days: 66.7%
  of those, within 30 days: 35.6%
  of those, within 60 days: 58.9%
  of those, within 90 days: 73.4%
Median days to 2nd order: 47


## 5. Headline numbers

In [6]:
fast, none = summary.loc["<=30 days"], summary.loc["No 2nd order"]
print(f"{fast.share_of_customers:.0%} of new customers ordered again within 30 days")
print(f"... and generated {fast.share_of_revenue:.0%} of the cohort's 6-month revenue")
print(f"Customers with no 2nd order: {none.share_of_customers:.0%} of customers, {none.share_of_revenue:.0%} of revenue")
print(f"Median first order: £{fast.median_first_order:,.0f} vs £{none.median_first_order:,.0f}")
print(f"Median 6-month revenue: £{fast.median_revenue_6m:,.0f} vs £{none.median_revenue_6m:,.0f} "
      f"({fast.median_revenue_6m/none.median_revenue_6m:.1f}x)")

24% of new customers ordered again within 30 days
... and generated 53% of the cohort's 6-month revenue
Customers with no 2nd order: 33% of customers, 9% of revenue
Median first order: £314 vs £233
Median 6-month revenue: £1,515 vs £233 (6.5x)


## 6. Export numbers for the figure

In [7]:
import json
out = {
    "cohort_size": int(len(cohort)),
    "share_of_customers": summary["share_of_customers"].round(6).to_dict(),
    "share_of_revenue": summary["share_of_revenue"].round(6).to_dict(),
    "median_revenue_6m": summary["median_revenue_6m"].round(0).to_dict(),
    "median_first_order": summary["median_first_order"].round(0).to_dict(),
}
Path("../figures/summary.json").write_text(json.dumps(out, indent=2))

575

## Limitations

- **Correlation, not causation.** Customers who reorder fast may simply be stronger buyers from the start. The analysis shows *where* to test (days 1 to 30), not that a nudge will work. The next step is a controlled experiment, e.g. a day-14 reminder vs. a holdout group.
- **B2B-heavy customer base.** Many customers of this retailer are wholesalers, so order values and frequency are higher than in typical B2C e-commerce.
- **Revenue, not margin.** The dataset has no cost data.
- **Single retailer, 2010 to 2011.** Patterns should be validated on a current, category-specific dataset.